# Data Storytelling Project Notebook

Brandon Kung 

Things to do:
- Redo the extraction of the data so you filter out non-kinetic missions, those with missing geographic coordinates, and those that INTERSECT with the Laotian border
- Revise the comments
- Remove "National" observation from the subnational HDI dataset

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

%matplotlib inline
%config InlineBackend.figure_format = "retina"

# Loading in the Subnational HDI Dataset and Exploration 

In [ ]:
shdi_df = pd.read_csv('../data/GDL-Subnational-HDI-data.csv')

In [27]:
shdi_df.head()

,Country,Continent,ISO_Code,Level,GDLCODE,Region,1990,1991,1992,1993,...,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023
0,Cambodia,Asia/Pacific,KHM,National,KHMt,Total,0.387,0.392,0.396,0.399,...,0.557,0.562,0.570,0.578,0.586,0.593,0.595,0.594,0.602,0.606
1,Cambodia,Asia/Pacific,KHM,Subnat,KHMr101,Banteay Mean Chey,0.372,0.377,0.380,0.383,...,0.564,0.573,0.586,0.598,0.610,0.621,0.626,0.630,0.638,0.642
2,Cambodia,Asia/Pacific,KHM,Subnat,KHMr113,Bat Dambang-Krong Pailin,0.396,0.401,0.405,0.407,...,0.581,0.582,0.586,0.590,0.593,0.596,0.594,0.589,0.596,0.600
3,Cambodia,Asia/Pacific,KHM,Subnat,KHMr102,Kampong Cham (incl Tboung Khmum),0.372,0.376,0.380,0.382,...,0.535,0.536,0.540,0.543,0.546,0.548,0.545,0.540,0.547,0.551
4,Cambodia,Asia/Pacific,KHM,Subnat,KHMr103,Kampong Chhnang,0.351,0.355,0.359,0.361,...,0.541,0.551,0.563,0.576,0.588,0.599,0.605,0.609,0.617,0.621


In [ ]:
shdi_df.info()

The Lao data has more subnational units and it was bombed more, so we will focus on Lao.

In [ ]:
# filtering dataset to include only Laotian provinces
lao_shdi = shdi_df[(shdi_df['ISO_Code'] == 'LAO') & (shdi_df['Level'] != 'National')]
lao_shdi['Level'].value_counts()

In [ ]:
id_cols = ['Country', 'Continent', 'ISO_Code', 'Level', 'GDLCODE', 'Region']

lao_shdi_df = pd.melt(
    lao_shdi,
    id_vars=id_cols,
    var_name='year',
    value_name='shdi_value'  
)

In [28]:
print(lao_shdi_df.columns)

Index(['Country', 'Continent', 'ISO_Code', 'Level', 'GDLCODE', 'Region',
       'year', 'shdi_value'],
      dtype='object')


In [29]:
# filtering to the columns we want
lao_shdi_df = lao_shdi_df[['ISO_Code', 'year', 'shdi_value']]

In [30]:
lao_shdi_df.head(20)

,ISO_Code,year,shdi_value
0,LA-AT,1990,0.379
1,LA-BK,1990,0.380
2,LA-BL,1990,0.453
3,LA-CH,1990,0.410
4,LA-HO,1990,0.394
5,LA-KH,1990,0.386
6,LA-LM,1990,0.399
7,LA-LP,1990,0.386
8,LA-OU,1990,0.357
9,LA-PH,1990,0.335


It seems like the ISO Code is not mapped by subnational divisions, so let's fix that.

In [ ]:
ISO_dict = {'Attapeu': 'LA-AT', 'Bokeo': 'LA-BK', 'Borikhamxay': 'LA-BL', 'Champasack': 'LA-CH', 
            'Huaphanh': 'LA-HO', 'Khammuane': 'LA-KH', 'Luangnamtha': 'LA-LM', 'Luangprabang': 'LA-LP', 'Oudomxay': 'LA-OU', 'Phongsaly': 'LA-PH', 
            'Saravane': 'LA-SL', 'Savannakhet': 'LA-SV', 'Sayabury': 'LA-XA', 'Sekong': 'LA-XE', 'Vientiane Municipality': 'LA-VT', 
           'Vientiane Province': 'LA-VI', 'Xiengkhuang': 'LA-XI'}

# Add the new column
lao_shdi_df['ISO_Code'] = lao_shdi_df['Region'].map(ISO_dict)
lao_shdi_df['ISO_Code'].head(17)

In [ ]:
# Graphing line trend for exploration purposes
plt.figure(figsize=(20, 6))

for key, grp in lao_shdi_df.groupby(['GDLCODE']):
    plt.plot(grp['year'], grp['shdi_value'], alpha=0.05, color='blue')

plt.title('Trajectories of Subnational HDI Over Time')
plt.xlabel('Year')
plt.ylabel('Subnational HDI Score')

# Loading in the THOR Bombing Dataset

In [ ]:
# import polars as pl

In [ ]:
# using polars to filter the data because it's very large 
#columns_to_keep = [
    #'TGTCOUNTRY',
    #'TGTLATDD_DDD_WGS84',
    #'TGTLONDDD_DDD_WGS84',
    #'MSNDATE',
    #'WEAPONTYPEWEIGHT',
    #'NUMWEAPONSDELIVERED',
    #'MFUNC_DESC_CLASS',
    #'WEAPONTYPECLASS',
    #'WEAPONTYPE'
#]

#(
    #pl.scan_csv('../data/large_thor_file.csv', ignore_errors=True)
    #.select(columns_to_keep)
    #.filter(pl.col('TGTCOUNTRY').str.to_uppercase() == 'LAOS')
    #.collect()
    #.write_csv('laos_bombing_clean.csv')
#)

In [ ]:
laos_df = pd.read_csv('../data/laos_bombing_clean.csv')

In [ ]:
laos_df.head()

In [ ]:
print(laos_df['MFUNC_DESC_CLASS'].value_counts())

In [ ]:
laos_df_clean = laos_df[laos_df['MFUNC_DESC_CLASS'].isin(['KINETIC'])]
laos_df_clean.head()

In [ ]:
laos_df_clean = laos_df_clean.dropna(subset=['TGTLATDD_DDD_WGS84', 'TGTLONDDD_DDD_WGS84'])
laos_df_clean.isna().sum()

In [ ]:
laos_df_clean

# Integrating Level 1 Boundaries (Provinces)

In [31]:
# Loading the administrative boundaries
gdf_provinces = gpd.read_file('../data/LaosBoundaryLvl1.json')
print(gdf_provinces.head())
gdf_provinces.columns

     GID_1 GID_0 COUNTRY       NAME_1                         VARNAME_1  \
0  LAO.1_1   LAO    Laos       Attapu  Attopu|Atpu|Attapeu|Attopeu|Muan   
1  LAO.2_1   LAO    Laos        Bokeo                                NA   
2  LAO.3_1   LAO    Laos  Bolikhamxai  Bolikhamsai|Bolikhamxay|Borikham   
3  LAO.4_1   LAO    Laos    Champasak  Bassac|Champassack|Champassak|Ch   
4  LAO.5_1   LAO    Laos     Houaphan     HuaPhan|Huaphanh|SamNeua|XamN   

  NL_NAME_1   TYPE_1 ENGTYPE_1 CC_1 HASC_1  ISO_1  \
0        NA  Khoueng  Province   NA  LA.AT  LA-AT   
1        NA  Khoueng  Province   NA  LA.BK  LA-BK   
2        NA  Khoueng  Province   NA  LA.BL  LA-BL   
3        NA  Khoueng  Province   NA  LA.CH  LA-CH   
4        NA  Khoueng  Province   NA  LA.HO     NA   

                                            geometry  
0  MULTIPOLYGON (((107.1091 14.394, 107.1061 14.3...  
1  MULTIPOLYGON (((100.7467 19.962, 100.7339 19.9...  
2  MULTIPOLYGON (((104.0322 18.8128, 104.0392 18....  
3  MULTIPO

Index(['GID_1', 'GID_0', 'COUNTRY', 'NAME_1', 'VARNAME_1', 'NL_NAME_1',
       'TYPE_1', 'ENGTYPE_1', 'CC_1', 'HASC_1', 'ISO_1', 'geometry'],
      dtype='object')

In [32]:
# Getting rid of useless columns
gdf_provinces = gdf_provinces[['GID_1', 'NAME_1', 'VARNAME_1', 'ENGTYPE_1', 'ISO_1', 'geometry']]
gdf_provinces

,GID_1,NAME_1,VARNAME_1,ENGTYPE_1,ISO_1,geometry
0,LAO.1_1,Attapu,Attopu|Atpu|Attapeu|Attopeu|Muan,Province,LA-AT,"MULTIPOLYGON (((107.1091 14.394, 107.1061 14.3..."
1,LAO.2_1,Bokeo,NA,Province,LA-BK,"MULTIPOLYGON (((100.7467 19.962, 100.7339 19.9..."
2,LAO.3_1,Bolikhamxai,Bolikhamsai|Bolikhamxay|Borikham,Province,LA-BL,"MULTIPOLYGON (((104.0322 18.8128, 104.0392 18...."
3,LAO.4_1,Champasak,Bassac|Champassack|Champassak|Ch,Province,LA-CH,"MULTIPOLYGON (((105.7842 14.0456, 105.785 14.0..."
4,LAO.5_1,Houaphan,HuaPhan|Huaphanh|SamNeua|XamN,Province,NA,"MULTIPOLYGON (((104.0044 20.9067, 104.021 20.9..."
5,LAO.6_1,Khammouan,Khammouane|Khammuan,Province,LA-KH,"MULTIPOLYGON (((105.7024 17.1156, 105.6997 17...."
6,LAO.7_1,LouangNamtha,Haut-Mekong|Luangnamtha|MuongLu,Province,LA-LM,"MULTIPOLYGON (((101.2223 20.3108, 101.2103 20...."
7,LAO.8_1,Louangphrabang,LoangPrabang|Louangphabang|Loua,Province,NA,"MULTIPOLYGON (((102.3773 19.3819, 102.373 19.3..."
8,LAO.9_1,Oudômxai,Oudomsai|Oudomsay|Oudomxay|UdomX,Province,NA,"MULTIPOLYGON (((102.1481 20.0885, 102.1318 20...."
9,LAO.10_1,Phôngsali,FongSali|Phongsaly,Province,NA,"MULTIPOLYGON (((102.624 20.9541, 102.5995 20.9..."


In [ ]:
# Setting up the plotting canvas
fig, ax = plt.subplots(figsize=(10, 10))

# Plotting the GeoDataFrame
gdf_provinces.plot(
    ax=ax, 
    facecolor='whitesmoke', 
    edgecolor='black', 
    linewidth=0.8
)

# Formatting the map
plt.title('Laos Administrative Boundaries', fontsize=16, pad=20)
ax.axis('off')

# 6. Render the plot
plt.tight_layout()
plt.show()

# Merging the datasets

In [33]:
# Merging the HDI and administrative boundaries 
gdf_provinces.rename(columns={'ISO_1': 'ISO_Code'}, inplace=True)
shdi_lvl1_merge = pd.merge(gdf_provinces, lao_shdi_df, on='ISO_Code', how='left')
shdi_lvl1_merge.head()

,GID_1,NAME_1,VARNAME_1,ENGTYPE_1,ISO_Code,geometry,year,shdi_value
0,LAO.1_1,Attapu,Attopu|Atpu|Attapeu|Attopeu|Muan,Province,LA-AT,"MULTIPOLYGON (((107.1091 14.394, 107.1061 14.3...",1990,0.379
1,LAO.1_1,Attapu,Attopu|Atpu|Attapeu|Attopeu|Muan,Province,LA-AT,"MULTIPOLYGON (((107.1091 14.394, 107.1061 14.3...",1991,0.384
2,LAO.1_1,Attapu,Attopu|Atpu|Attapeu|Attopeu|Muan,Province,LA-AT,"MULTIPOLYGON (((107.1091 14.394, 107.1061 14.3...",1992,0.389
3,LAO.1_1,Attapu,Attopu|Atpu|Attapeu|Attopeu|Muan,Province,LA-AT,"MULTIPOLYGON (((107.1091 14.394, 107.1061 14.3...",1993,0.395
4,LAO.1_1,Attapu,Attopu|Atpu|Attapeu|Attopeu|Muan,Province,LA-AT,"MULTIPOLYGON (((107.1091 14.394, 107.1061 14.3...",1994,0.403


In [34]:
# Convert the pandas DataFrame into a GeoDataFrame of points
# WGS84 (EPSG:4326) is the standard CRS for latitude/longitude coordinates
bombing_gdf = gpd.GeoDataFrame(
    laos_df_clean, 
    geometry=gpd.points_from_xy(laos_df_clean['TGTLONDDD_DDD_WGS84'], laos_df_clean['TGTLATDD_DDD_WGS84']),
    crs="EPSG:4326"
)

# 3. Align CRS
if bombing_gdf.crs != shdi_lvl1_merge.crs:
    shdi_lvl1_merge = shdi_lvl1_merge.to_crs(bombing_gdf.crs)

# 4. Perform the Spatial Join
# 'predicate="within"' ensures we find which province polygon each point sits inside
# 'how="left"' keeps all points and appends the province columns to them
points_with_provinces = gpd.sjoin(bombing_gdf, shdi_lvl1_merge, how='left', predicate='within')

# 5. Group by Province and Aggregate the Impact
# Replace 'NAME_1' with the actual column name for province names in your shapefile
spatial_summary = points_with_provinces.groupby('NAME_1').agg(
    total_missions=('TGTCOUNTRY', 'size'),
    total_tonnage=('WEAPONTYPEWEIGHT', 'sum')
).reset_index()

# View the result
print(spatial_summary.head())

        NAME_1  total_missions  total_tonnage
0       Attapu         1981690      519443704
1        Bokeo            7378         657526
2  Bolikhamxai          226406       70311184
3    Champasak          441422       40669202
4     Houaphan           21609        7507176


In [36]:
# Merge the aggregated statistics back to the geometry data
bombing_admin_df = shdi_lvl1_merge.merge(spatial_summary, on='NAME_1', how='left')

# Fill provinces that had zero bombing missions with 0 instead of NaN
bombing_admin_df['total_tonnage'] = bombing_admin_df['total_tonnage'].fillna(0)
bombing_admin_df['total_missions'] = bombing_admin_df['total_missions'].fillna(0)

In [37]:
bombing_admin_df.head()

,GID_1,NAME_1,VARNAME_1,ENGTYPE_1,ISO_Code,geometry,year,shdi_value,total_missions,total_tonnage
0,LAO.1_1,Attapu,Attopu|Atpu|Attapeu|Attopeu|Muan,Province,LA-AT,"MULTIPOLYGON (((107.1091 14.394, 107.1061 14.3...",1990,0.379,1981690,519443704
1,LAO.1_1,Attapu,Attopu|Atpu|Attapeu|Attopeu|Muan,Province,LA-AT,"MULTIPOLYGON (((107.1091 14.394, 107.1061 14.3...",1991,0.384,1981690,519443704
2,LAO.1_1,Attapu,Attopu|Atpu|Attapeu|Attopeu|Muan,Province,LA-AT,"MULTIPOLYGON (((107.1091 14.394, 107.1061 14.3...",1992,0.389,1981690,519443704
3,LAO.1_1,Attapu,Attopu|Atpu|Attapeu|Attopeu|Muan,Province,LA-AT,"MULTIPOLYGON (((107.1091 14.394, 107.1061 14.3...",1993,0.395,1981690,519443704
4,LAO.1_1,Attapu,Attopu|Atpu|Attapeu|Attopeu|Muan,Province,LA-AT,"MULTIPOLYGON (((107.1091 14.394, 107.1061 14.3...",1994,0.403,1981690,519443704


# Integrating District Boundaries

In [ ]:
# Loading district boundaries
gdf_districts = gpd.read_file('../data/LaosBoundaryLvl2.json')
print(gdf_districts.head())

# 3. Set up the plotting canvas
fig, ax = plt.subplots(figsize=(10, 10))

# 4. Plot the GeoDataFrame
# - facecolor: fills the polygons. 'whitesmoke' gives a nice base map feel.
# - edgecolor: color of the province borders.
# - linewidth: thickness of the borders.
gdf_districts.plot(
    ax=ax, 
    facecolor='whitesmoke', 
    edgecolor='black', 
    linewidth=0.8
)

# 5. Formatting the map
plt.title('Laos Administrative Boundaries', fontsize=16, pad=20)

# Turn off the latitude/longitude axis labels for a cleaner, map-like appearance
ax.axis('off')

# 6. Render the plot
plt.tight_layout()
plt.show()